# Graph Constant Bottleneck — katbailey/factorizer

**Smell (`factorizer.py` lines 142, 170, 173):** User/item factor matrices are wrapped in `tf.constant()`, embedding the entire dataset permanently into the TF graph. These tensors cannot be garbage-collected for the lifetime of the session, pinning large arrays in memory.

**Fix:** Use `tf.Variable` initialised from numpy data (mutable, releasable) and feed data through a `tf.data.Dataset` pipeline instead.

In [ ]:
!pip install -q codecarbon

In [ ]:
import codecarbon, os, json
_dir = os.path.join(os.path.dirname(codecarbon.__file__), 'data', 'private_infra')
os.makedirs(_dir, exist_ok=True)
_p = os.path.join(_dir, 'nordic_emissions.json')
if not os.path.exists(_p):
    with open(_p, 'w') as f: json.dump({}, f)
print('CodeCarbon ready')

In [ ]:
import gc, os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import tensorflow as tf
from codecarbon import EmissionsTracker

os.makedirs('results', exist_ok=True)
N_RUNS   = 10
N_ITER   = 500   # gradient descent iterations
N_USERS  = 2000
N_ITEMS  = 1000
N_FACTORS= 20
LR       = 0.01
REG      = 0.1

# Generate synthetic ratings matrix (sparse — only ~1% filled)
np.random.seed(42)
n_ratings = N_USERS * N_ITEMS // 100
user_ids  = np.random.randint(0, N_USERS, n_ratings).astype(np.int32)
item_ids  = np.random.randint(0, N_ITEMS, n_ratings).astype(np.int32)
ratings   = np.random.uniform(1, 5, n_ratings).astype(np.float32)

# Dense rating matrix for constant-embedding demonstration
R_dense = np.zeros((N_USERS, N_ITEMS), dtype=np.float32)
R_dense[user_ids, item_ids] = ratings

print(f'Synthetic ratings: {n_ratings} entries  matrix shape: {R_dense.shape}')

## BEFORE — Smell Active (tf.constant embeds full matrix into graph)

In [ ]:
results_before = []

for run in range(1, N_RUNS + 1):
    print(f'\n[BEFORE] Run {run}/{N_RUNS}')
    tf.keras.backend.clear_session()

    tracker = EmissionsTracker(
        project_name=f'factorizer_before_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()

    # ❌ SMELL (mirrors factorizer.py lines 142/170/173):
    # Entire rating matrix embedded as tf.constant — permanently pinned in the graph.
    R_const = tf.constant(R_dense, dtype=tf.float32)   # ← smell

    W = tf.Variable(tf.random.normal([N_USERS, N_FACTORS], stddev=0.01))
    H = tf.Variable(tf.random.normal([N_FACTORS, N_ITEMS], stddev=0.01))
    opt = tf.optimizers.SGD(LR)

    for _ in range(N_ITER):
        with tf.GradientTape() as tape:
            pred  = tf.matmul(W, H)
            mask  = tf.cast(R_const > 0, tf.float32)
            loss  = tf.reduce_sum(mask * tf.square(R_const - pred)) + \
                    REG * (tf.reduce_sum(tf.square(W)) + tf.reduce_sum(tf.square(H)))
        grads = tape.gradient(loss, [W, H])
        opt.apply_gradients(zip(grads, [W, H]))

    final_loss = loss.numpy()
    tracker.stop()
    em = tracker.final_emissions_data
    results_before.append({
        'run': run, 'final_loss': round(float(final_loss), 4),
        'energy_kWh': round(em.energy_consumed,8),
        'cpu_energy_kWh': round(em.cpu_energy,8),
        'gpu_energy_kWh': round(em.gpu_energy,8),
        'ram_energy_kWh': round(em.ram_energy,8),
        'emissions_kgCO2': round(em.emissions,8),
        'duration_s': round(em.duration,2),
    })
    print(f'  loss={final_loss:.2f}  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    del R_const, W, H, opt
    gc.collect()

df_before = pd.DataFrame(results_before)
df_before.to_csv('results/katbailey_factorizer_before.csv', index=False)
print('\n--- BEFORE means ---')
print(df_before.mean(numeric_only=True))
print('Saved results/katbailey_factorizer_before.csv')

## AFTER — Smell Fixed (tf.data.Dataset pipeline — no full-matrix constant)

In [ ]:
results_after = []

BATCH = 512

for run in range(1, N_RUNS + 1):
    print(f'\n[AFTER] Run {run}/{N_RUNS}')
    tf.keras.backend.clear_session()
    gc.collect()

    tracker = EmissionsTracker(
        project_name=f'factorizer_after_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()

    # ✅ FIX: stream triplets through tf.data — no full matrix pinned in graph
    ds = tf.data.Dataset.from_tensor_slices(
        (user_ids, item_ids, ratings)
    ).shuffle(10000).batch(BATCH).repeat()
    ds_iter = iter(ds)

    W = tf.Variable(tf.random.normal([N_USERS, N_FACTORS], stddev=0.01))
    H = tf.Variable(tf.random.normal([N_FACTORS, N_ITEMS], stddev=0.01))
    opt = tf.optimizers.SGD(LR)

    steps_per_run = N_ITER
    for _ in range(steps_per_run):
        u_batch, i_batch, r_batch = next(ds_iter)
        with tf.GradientTape() as tape:
            w_u  = tf.gather(W, u_batch)
            h_i  = tf.gather(tf.transpose(H), i_batch)
            pred = tf.reduce_sum(w_u * h_i, axis=1)
            loss = tf.reduce_mean(tf.square(r_batch - pred)) + \
                   REG * (tf.reduce_sum(tf.square(W)) + tf.reduce_sum(tf.square(H)))
        grads = tape.gradient(loss, [W, H])
        opt.apply_gradients(zip(grads, [W, H]))

    final_loss = loss.numpy()
    tracker.stop()
    em = tracker.final_emissions_data
    results_after.append({
        'run': run, 'final_loss': round(float(final_loss), 4),
        'energy_kWh': round(em.energy_consumed,8),
        'cpu_energy_kWh': round(em.cpu_energy,8),
        'gpu_energy_kWh': round(em.gpu_energy,8),
        'ram_energy_kWh': round(em.ram_energy,8),
        'emissions_kgCO2': round(em.emissions,8),
        'duration_s': round(em.duration,2),
    })
    print(f'  loss={final_loss:.2f}  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    del W, H, opt, ds, ds_iter
    gc.collect()

df_after = pd.DataFrame(results_after)
df_after.to_csv('results/katbailey_factorizer_after.csv', index=False)
print('\n--- AFTER means ---')
print(df_after.mean(numeric_only=True))
print('Saved results/katbailey_factorizer_after.csv')

In [ ]:
print('\n=== SUMMARY: katbailey/factorizer ===')
print(f"BEFORE avg energy : {df_before['energy_kWh'].mean():.6f} kWh")
print(f"AFTER  avg energy : {df_after['energy_kWh'].mean():.6f} kWh")
print(f"BEFORE avg CO2    : {df_before['emissions_kgCO2'].mean():.6f} kg")
print(f"AFTER  avg CO2    : {df_after['emissions_kgCO2'].mean():.6f} kg")
print(f"BEFORE avg time   : {df_before['duration_s'].mean():.1f} s")
print(f"AFTER  avg time   : {df_after['duration_s'].mean():.1f} s")